## ----------------- Movie DM -------------------

## 1. whole-brain group results

process for whole model & each movie feature

In [ ]:
import os
import glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import multipletests
from nilearn import plotting
from pathlib import Path

# ------------------------- CONFIGURATION -------------------------
MASK_FILE = "MNI152_T1_1mm_brain_resampled_HBN_brainmask.nii.gz"
ROOT_FOLDER = Path("movieDM_lexisurp_10feature_result/")

OUTPUT_FOLDER = Path("1_movieDM_wholebrain/")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

CONS = ["whole_model", "GPT3_surprisal", "Lg10WF", "spoken_words", "written_words", 
        "positive", "negative", "faces", "body", "brightness", "loudness"]
ALPHA = 0.05
CORRECTION_METHOD = "bonf"  

# ------------------------- LOAD BRAIN MASK -------------------------
mask_img = nib.load(MASK_FILE)
gray_mask = mask_img.get_fdata()
mask_indices = np.where(gray_mask > 0)
num_voxels = len(mask_indices[0])
volume_shape = gray_mask.shape
space_size = np.prod(volume_shape)

# ------------------------- HELPER FUNCTIONS -------------------------
def save_nifti_from_voxelwise_data(data_1d, filename, title=None, plot=True):
    """Save and optionally plot brain data using the brain mask."""
    full_data = np.zeros(space_size)
    full_data[gray_mask.reshape(space_size) > 0] = data_1d
    full_data = full_data.reshape(volume_shape)
    img = nib.Nifti1Image(full_data, mask_img.affine, mask_img.header)
    img.to_filename(filename)
    if plot:
        plotting.plot_glass_brain(img, title=title, display_mode='lzry', colorbar=True, plot_abs=False)
        plt.savefig(str(filename).replace(".nii.gz", ".png"))
        plt.close()
    return img

def load_subject_data(file_list):
    """Load subject-wise data into (n_voxels, n_subjects) matrix."""
    data = np.zeros((num_voxels, len(file_list)))
    for i, file in enumerate(file_list):
        img = nib.load(file)
        data[:, i] = img.get_fdata()[mask_indices]
    return data

# ------------------------- MAIN LOOP -------------------------

for con in CONS:  
    print(f"\nProcessing contrast: {con}")
    
    # 1. Find relevant subject files
    pattern = ROOT_FOLDER / "sub-*" / f"*{con}*.nii.gz"
    file_list = sorted(glob.glob(str(pattern)))
    n_subjects = len(file_list)
    print(f"Found {n_subjects} subject files.")

    # 2. Load subject data
    all_subject_data = load_subject_data(file_list)
    print("Loaded data shape:", all_subject_data.shape)

    # 3. Save mean across subjects
    group_mean = np.mean(all_subject_data, axis=1)
    save_nifti_from_voxelwise_data(
        group_mean,
        OUTPUT_FOLDER / f"{con}_group_mean_n{n_subjects}.nii.gz",
        title=f"{con}: Mean across subjects"
    )

    # 4. T-test across subjects (per voxel)
    t_stats, p_values = ttest_1samp(all_subject_data, popmean=0, axis=1)
    print("T-test complete. Saving unthresholded t-map.")
    save_nifti_from_voxelwise_data(
        t_stats,
        OUTPUT_FOLDER / f"{con}_tstats_n{n_subjects}_NOthres.nii.gz",
        title=f"{con}: T-stats (uncorrected)"
    )

    # 5. Multiple comparison correction
    print(f"Applying multiple comparison correction: {CORRECTION_METHOD.upper()}")
    rejected, pvals_corrected, _, _ = multipletests(p_values, alpha=ALPHA, method=CORRECTION_METHOD)

    # 6. Save corrected t-stats
    corrected_t_stats = np.zeros_like(t_stats)
    corrected_t_stats[rejected] = t_stats[rejected]
    save_nifti_from_voxelwise_data(
        corrected_t_stats,
        OUTPUT_FOLDER / f"{con}_tstats_n{n_subjects}_{CORRECTION_METHOD.upper()}_alpha{ALPHA}.nii.gz",
        title=f"{con}: Corrected T-stats ({CORRECTION_METHOD.upper()})"
    )


## 2. network-level results using unthresholded ROI maps

This analysis extracts lexical surprisal encoding score from unthresholded group-level ROI maps that resampled to the HBN fMRI data resolution. 
The ROI sets (available in 'unthres_ROI') include:

- **10 Language network ROIs** (labeled as 'CoreLanguage')
- **20 Multiple demand (MD) network ROIs** that do not overlap with the language-network ROIs
- **10 Theory of mind (ToM) ROIs** that do not overlap with the language-network ROIs

Sources of the original parcels:
- Language and MD parcels: https://www.evlab.mit.edu/resources-all/download-parcels
- ToM parcels: https://osf.io/bzwm8/overview

### a. get network map dictionary

In [ ]:
from collections import defaultdict
import glob
import os

# ROIs
ROI_dir = 'unthres_ROI/'
ROIs = sorted(glob.glob(os.path.join(ROI_dir, 'rROI*.nii')))
ROI_names = ["_".join(os.path.splitext(roi)[0].split('_')[-2:]) for roi in ROIs]
print(len(ROIs))

# ---------- STEP 0: Define Network Membership ----------
network_map = defaultdict(list)
roi_network_labels = []

for roi_path in ROIs:
    
    fname = os.path.basename(roi_path).replace('.nii', '')
    roi_name = "_".join(os.path.splitext(roi_path)[0].split('_')[-2:])
    
    # Split into network and ROI name (e.g., MDN_LPostParietal → MDN, LPostParietal)
    if "_" in fname:
        _, net, roi = fname.split("_", 2)

    else:
        continue

    # Determine hemisphere based on ROI prefix
    if roi.startswith("L"):
        hemi = "LH"
    elif roi.startswith("R"):
        hemi = "RH"

    # Save labels
    roi_network_labels.append((roi_name, net, hemi))

    # Include whole network, hemisphere-specific network, and ROI
    network_map[f"Whole_{net}"].append(roi_name)
    network_map[f"{hemi}_{net}"].append(roi_name)

print(network_map)

desired_column_order = [
    "ID",
    "Whole_CoreLanguage",
    "LH_CoreLanguage",
    "RH_CoreLanguage",
    "CoreLanguage_LIFGorb",
    "CoreLanguage_LIFG",
    "CoreLanguage_LMFG",
    "CoreLanguage_LAntTemp",
    "CoreLanguage_LPostTemp",
    "CoreLanguage_RIFGorb",
    "CoreLanguage_RIFG",
    "CoreLanguage_RMFG",
    "CoreLanguage_RAntTemp",
    "CoreLanguage_RPostTemp",
    "Whole_MDN",
    "LH_MDN",
    "RH_MDN",
    "MDN_LPostParietal",
    "MDN_LMidParietal",
    "MDN_LAntParietal",
    "MDN_LSFG",
    "MDN_LPrecG",
    "MDN_LIFGop",
    "MDN_LMFG",
    "MDN_LMFGorb",
    "MDN_LInsula",
    "MDN_LMedialFrontal",
    "MDN_RPostParietal",
    "MDN_RMidParietal",
    "MDN_RAntParietal",
    "MDN_RSFG",
    "MDN_RPrecG",
    "MDN_RIFGop",
    "MDN_RMFG",
    "MDN_RMFGorb",
    "MDN_RInsula",
    "MDN_RMedialFrontal",
    "Whole_ToM",
    "LH_ToM",
    "RH_ToM",
    "ToM_LTPJ",
    "ToM_LDMPFC",
    "ToM_LMMPFC",
    "ToM_LVMPFC",
    "ToM_LPC",
    "ToM_RTPJ",
    "ToM_RDMPFC",
    "ToM_RMMPFC",
    "ToM_RVMPFC",
    "ToM_RPC"
]

### b. extract roi-level and network-level surprisal encoding

In [ ]:
import glob
import os
from pathlib import Path
import nibabel as nib
import numpy as np
import pandas as pd
from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import fdrcorrection

# ROIs
ROI_dir = 'unthres_ROI/'
ROIs = sorted(glob.glob(os.path.join(ROI_dir, 'rROI*.nii')))
ROI_names = ["_".join(os.path.splitext(roi)[0].split('_')[-2:]) for roi in ROIs]

ROOT_FOLDER = Path("movieDM_lexisurp_10feature_result/") #where the subject-level contrast maps are located

CONS = ["GPT3_surprisal"]

OUTPUT_FOLDER = Path("2_movieDM_ROIresults/")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

ttest_results = []

for con in CONS:
    
    print(f"\nProcessing contrast: {con}")
    
    pattern = ROOT_FOLDER / "sub-*" / f"*{con}*.nii.gz"
    sub_maps = sorted(glob.glob(str(pattern)))
    n_subjects = len(sub_maps)
    print(f"Found {n_subjects} subject files.")
    
    # ---------- STEP 1: Compute ROI & Network-Level Stats ----------
    roi_values_all = {roi_name: [] for roi_name in ROI_names}

    for subimg in sub_maps:
        z_img = nib.load(subimg)
        z_data = z_img.get_fdata()

        for roi_path, roi_name in zip(ROIs, ROI_names):
            roi_mask = nib.load(roi_path).get_fdata()
            roi_voxels = z_data[roi_mask > 0]

            if len(roi_voxels) > 0:
                mean_val = np.nanmean(roi_voxels) #mean values
            else:
                mean_val = np.nan

            roi_values_all[roi_name].append(mean_val)

    df_results = pd.DataFrame(roi_values_all)  
    df_results.insert(0, "ID", [os.path.basename(p).split("_")[0].replace('sub-', '') for p in sub_maps])
    
    for net_label, roi_list in network_map.items():
        df_results[net_label] = df_results[roi_list].mean(axis=1)

    # re-arrange order
    df_results = df_results[desired_column_order]
    df_results.to_csv(os.path.join(OUTPUT_FOLDER, f"movieDM_{con}_ROImean_n{n_subjects}.csv"), index=False)
    
    # ---------- STEP 2: Perform one-sample t-tests for each Network/ROI ----------
    for col in df_results.columns[1:]:
        values = df_results[col].dropna()
        
        # Compute mean and SEM
        mean_val = values.mean()
        sem_val = values.std(ddof=1) / np.sqrt(len(values))
        
        # Perform one-sample t-test
        t_stat, p_val = ttest_1samp(values, 0.0, nan_policy='omit')
        
        # Append results
        ttest_results.append({
            'Contrast': con,
            'Network/ROI': col,
            'Mean': mean_val,
            'SEM': sem_val,
            'T_stat': t_stat,
            'p_uncorrected': p_val
        })

    # 3. Create DataFrame from results
    df_ttest_results = pd.DataFrame(ttest_results)

    # FDR correction
    rej_net, pvals_fdr_net = fdrcorrection(df_ttest_results["p_uncorrected"])
    df_ttest_results["p_fdr_corrected"] = pvals_fdr_net
    df_ttest_results["significant"] = rej_net
    
    df_ttest_results.to_csv(os.path.join(OUTPUT_FOLDER, f"movieDM_{con}_ROImean_n{n_subjects}_ttest.csv"), index=False)

## ----------------- Movie TP -------------------

## 1. whole-brain group results

In [ ]:
import os
import glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import multipletests
from nilearn import plotting
from pathlib import Path

# ------------------------- CONFIGURATION -------------------------
MASK_FILE = "MNI152_T1_1mm_brain_resampled_HBN_brainmask.nii.gz"
ROOT_FOLDER = Path("movieTP_lexisurp_10feature_crossresult/")

OUTPUT_FOLDER = Path("1_movieTP_wholebrain/")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

CONS = ["whole_model", "GPT3_surprisal", "Lg10WF", "spoken_words", "written_words", 
        "positive", "negative", "faces", "body", "brightness", "loudness"]
ALPHA = 0.05
CORRECTION_METHOD = "bonf"  

# ------------------------- LOAD BRAIN MASK -------------------------
mask_img = nib.load(MASK_FILE)
gray_mask = mask_img.get_fdata()
mask_indices = np.where(gray_mask > 0)
num_voxels = len(mask_indices[0])
volume_shape = gray_mask.shape
space_size = np.prod(volume_shape)

# ------------------------- HELPER FUNCTIONS -------------------------
def save_nifti_from_voxelwise_data(data_1d, filename, title=None, plot=True):
    """Save and optionally plot brain data using the brain mask."""
    full_data = np.zeros(space_size)
    full_data[gray_mask.reshape(space_size) > 0] = data_1d
    full_data = full_data.reshape(volume_shape)
    img = nib.Nifti1Image(full_data, mask_img.affine, mask_img.header)
    img.to_filename(filename)
    if plot:
        plotting.plot_glass_brain(img, title=title, display_mode='lzry', colorbar=True, plot_abs=False)
        plt.savefig(str(filename).replace(".nii.gz", ".png"))
        plt.close()
    return img

def load_subject_data(file_list):
    """Load subject-wise data into (n_voxels, n_subjects) matrix."""
    data = np.zeros((num_voxels, len(file_list)))
    for i, file in enumerate(file_list):
        img = nib.load(file)
        data[:, i] = img.get_fdata()[mask_indices]
    return data

# ------------------------- MAIN LOOP -------------------------

for con in CONS:  
    print(f"\nProcessing contrast: {con}")
    
    # 1. Find relevant subject files
    pattern = ROOT_FOLDER / "sub-*" / f"*{con}*.nii.gz"
    file_list = sorted(glob.glob(str(pattern)))
    n_subjects = len(file_list)
    print(f"Found {n_subjects} subject files.")

    # 2. Load subject data
    all_subject_data = load_subject_data(file_list)
    print("Loaded data shape:", all_subject_data.shape)

    # 3. Save mean across subjects
    group_mean = np.mean(all_subject_data, axis=1)
    save_nifti_from_voxelwise_data(
        group_mean,
        OUTPUT_FOLDER / f"{con}_group_mean_n{n_subjects}.nii.gz",
        title=f"{con}: Mean across subjects"
    )

    # 4. T-test across subjects (per voxel)
    t_stats, p_values = ttest_1samp(all_subject_data, popmean=0, axis=1)
    print("T-test complete. Saving unthresholded t-map.")
    save_nifti_from_voxelwise_data(
        t_stats,
        OUTPUT_FOLDER / f"{con}_tstats_n{n_subjects}_NOthres.nii.gz",
        title=f"{con}: T-stats (uncorrected)"
    )

    # 5. Multiple comparison correction
    print(f"Applying multiple comparison correction: {CORRECTION_METHOD.upper()}")
    rejected, pvals_corrected, _, _ = multipletests(p_values, alpha=ALPHA, method=CORRECTION_METHOD)

    # 6. Save corrected t-stats
    corrected_t_stats = np.zeros_like(t_stats)
    corrected_t_stats[rejected] = t_stats[rejected]
    save_nifti_from_voxelwise_data(
        corrected_t_stats,
        OUTPUT_FOLDER / f"{con}_tstats_n{n_subjects}_{CORRECTION_METHOD.upper()}_alpha{ALPHA}.nii.gz",
        title=f"{con}: Corrected T-stats ({CORRECTION_METHOD.upper()})"
    )


## 2. network-level results using participant-specific ROI (top 10% voxels from DM map)

This step uses the same network_map dictionary constructed in above DM analysis

In [ ]:
import os
import glob
from pathlib import Path
import nibabel as nib
import numpy as np
import pandas as pd
from tqdm import tqdm

# === File Paths ===
TP_root = Path("movieTP_lexisurp_10feature_crossresult/")
DM_root = Path("movieDM_lexisurp_10feature_result/")

ROI_dir = 'unthres_ROI/'
ROIs = sorted(glob.glob(os.path.join(ROI_dir, 'rROI*.nii')))
ROI_names = ["_".join(os.path.splitext(roi)[0].split('_')[-2:]) for roi in ROIs]

CON = "GPT3_surprisal"  # Or loop through CONS if needed

OUTPUT_FOLDER = Path("2_movieTP_ROIresults/")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# === Output dictionary ===
roi_values_all = {roi_name: [] for roi_name in ROI_names}
subject_ids = []

# === Iterate through TP subjects ===
TP_subject_dirs = sorted(TP_root.glob("*NDAR*"))
print("Total number of subjects:", len(TP_subject_dirs))

for tp_sub_dir in tqdm(TP_subject_dirs):
    sub_id = tp_sub_dir.name
    dm_sub_dir = DM_root / f"sub-{sub_id}"
    
    # Skip if no corresponding DM folder
    if not dm_sub_dir.exists():
        print("Warning: Not find files for - ",dm_sub_dir)
        continue

    # Load DM and TP zstat maps
    dm_file = dm_sub_dir / f"sub-{sub_id}_{CON}_mean_zscore.nii.gz"
    tp_file = tp_sub_dir / f"{sub_id}_{CON}_zscore.nii.gz"

    if not (dm_file.exists() and tp_file.exists()):
        print("Warning: Not find files for - ", dm_file)
        continue

    dm_data = nib.load(dm_file).get_fdata()
    tp_data = nib.load(tp_file).get_fdata()
    
    subject_ids.append(sub_id)

    for roi_path, roi_name in zip(ROIs, ROI_names):
        
        roi_mask = nib.load(roi_path).get_fdata()
        roi_coords = np.where(roi_mask > 0)

        if roi_coords[0].size == 0:
            print("Empty masks for ROI:", roi_name)
            roi_values_all[roi_name].append(np.nan)
            continue

        # --- Step 1: Get top 10% DM voxels based on DM map in this ROI ---
        dm_vals = dm_data[roi_coords]
        # add: deal with NaN values
        dm_vals = dm_vals[~np.isnan(dm_vals)]
        
        if len(dm_vals) == 0:
            print("Empty data for ROI:", roi_name)
            roi_values_all[roi_name].append(np.nan)
            continue

        top_k = int(np.ceil(len(dm_vals) * 0.1))
        top_indices = np.argsort(dm_vals)[-top_k:]

        top_voxel_coords = tuple([coord[top_indices] for coord in roi_coords])

        # --- Step 2: Get mean value of those voxels from TP data ---
        tp_vals = tp_data[top_voxel_coords]
        # add: deal with NaN values
        tp_vals = tp_vals[~np.isnan(tp_vals)]
        mean_tp_val = np.mean(tp_vals)

        # --- Step 3: Save result ---
        roi_values_all[roi_name].append(mean_tp_val)

df_results = pd.DataFrame(roi_values_all)  
df_results.insert(0, "ID", [os.path.basename(p).split("_")[0].replace('sub-', '') for p in sub_maps])
    
for net_label, roi_list in network_map.items():
    df_results[net_label] = df_results[roi_list].mean(axis=1)

# re-arrange order
df_results = df_results[desired_column_order]
df_results.to_csv(f"{OUTPUT_FOLDER}/{CON}_ROI_CrossV_Top10percDefinedDM_n{n_subjects}.csv",index=False)
    
# Perform one-sample t-tests for each Network/ROI
for col in df_results.columns[1:]:
    values = df_results[col].dropna()
        
    # Compute mean and SEM
    mean_val = values.mean()
    sem_val = values.std(ddof=1) / np.sqrt(len(values))
        
    # Perform one-sample t-test
    t_stat, p_val = ttest_1samp(values, 0.0, nan_policy='omit')
        
    # Append results
    ttest_results.append({
            'Contrast': con,
            'Network/ROI': col,
            'Mean': mean_val,
            'SEM': sem_val,
            'T_stat': t_stat,
            'p_uncorrected': p_val
        })

# 3. Create DataFrame from results
df_ttest_results = pd.DataFrame(ttest_results)

# FDR correction
rej_net, pvals_fdr_net = fdrcorrection(df_ttest_results["p_uncorrected"])
df_ttest_results["p_fdr_corrected"] = pvals_fdr_net
df_ttest_results["significant"] = rej_net
    
df_ttest_results.to_csv(f"{OUTPUT_FOLDER}/{CON}_ROI_CrossV_Top10percDefinedDM_n{n_subjects}_ttest.csv",index=False)

## 3. cross-movie similarity analysis

This step uses the same network_map dictionary constructed in above DM analysis

### a. within-subject cross-movie similarity

In [ ]:
import os
import glob
from pathlib import Path
import nibabel as nib
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from tqdm import tqdm

# === File Paths ===
TP_root = Path("movieTP_lexisurp_10feature_crossresult/")
DM_root = Path("movieDM_lexisurp_10feature_result/")

ROI_dir = 'unthres_ROI/'
ROIs = sorted(glob.glob(os.path.join(ROI_dir, 'rROI*.nii')))
ROI_names = ["_".join(os.path.splitext(roi)[0].split('_')[-2:]) for roi in ROIs]

CON = "GPT3_surprisal"

OUTPUT_FOLDER = Path("2_movieTP_ROIresults/cross_movie_similarity/")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# === Output dictionary ===
correlation_results = {roi_name: [] for roi_name in ROI_names}
subject_ids = []

# === Iterate through TP subjects ===
tp_subject_dirs = sorted(TP_root.glob("*NDAR*"))

for tp_sub_dir in tqdm(tp_subject_dirs):
    
    sub_id = tp_sub_dir.name
    dm_sub_dir = DM_root / f"sub-{sub_id}"
    
    # Skip if no corresponding DM folder
    if not dm_sub_dir.exists():
        continue

    # Load DM and TP zstat maps
    dm_file = dm_sub_dir / f"sub-{sub_id}_{CON}_mean_zscore.nii.gz"
    tp_file = tp_sub_dir / f"{sub_id}_{CON}_zscore.nii.gz"

    if not (dm_file.exists() and tp_file.exists()):
        continue

    dm_data = nib.load(dm_file).get_fdata()
    tp_data = nib.load(tp_file).get_fdata()
    
    subject_ids.append(sub_id)

    for roi_path, roi_name in zip(ROIs, ROI_names):
        
        roi_mask = nib.load(roi_path).get_fdata()
        roi_coords = np.where(roi_mask > 0)

        if roi_coords[0].size == 0:
            correlation_results[roi_name].append(np.nan)
            continue

        dm_vals = dm_data[roi_coords]
        dm_vals = np.nan_to_num(dm_vals, nan=0.0)  
        
        tp_vals = tp_data[roi_coords]
        tp_vals = np.nan_to_num(tp_vals, nan=0.0) 

        # Keep only indices where both x and y are not NaN
        mask = ~np.isnan(dm_vals) & ~np.isnan(tp_vals)

        if np.std(dm_vals) == 0 or np.std(tp_vals) == 0:
            corr = 0 # mark as 0 not nan
        else:
            corr, _ = np.arctanh(pearsonr(dm_vals[mask], tp_vals[mask])) # Fisher-z transformation

        correlation_results[roi_name].append(corr)

# === Build DataFrame ===
df_corr = pd.DataFrame(correlation_results)
df_corr.insert(0, "ID", subject_ids)

for net_label, roi_list in network_map.items():
       df_corr[net_label] = df_corr[roi_list].mean(axis=1)

# re-arrange order
df_corr = df_corr[desired_column_order] 
df_corr.to_csv(os.path.join(OUTPUT_FOLDER, f"{CON}_movieDMandTP_ROI_withinsubCrossMovieCorrelation_n{len(tp_subject_dirs)}.csv"), index=False)


### b. cross-subject within-movie similarity (as reference)

#### movie DM

In [ ]:
import os
import glob
import nibabel as nib
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr
from tqdm import tqdm

# === Set Paths ===
ROI_dir = 'unthres_ROI/'
ROIs = sorted(glob.glob(os.path.join(ROI_dir, 'rROI*.nii')))
ROI_names = ["_".join(os.path.splitext(roi)[0].split('_')[-2:]) for roi in ROIs]

ROOT_FOLDER = Path("movieDM_lexisurp_10feature_result/")
CON = "GPT3_surprisal"
pattern = ROOT_FOLDER / "sub-*" / f"*{CON}*.nii.gz"
sub_maps = sorted(glob.glob(str(pattern)))
n_subjects = len(sub_maps)
subject_ids = [os.path.basename(p).split("_")[0] for p in sub_maps]
print(f"Found {n_subjects} subject files.")

OUTPUT_FOLDER = Path("2_movieTP_ROIresults/cross_movie_similarity/")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# === Output container for ISC results ===
isc_dict = {}

# === Process Each ROI ===
for roi_path, roi_name in tqdm(zip(ROIs, ROI_names), total=len(ROIs), desc="Processing ROIs"):
    
    roi_mask = nib.load(roi_path).get_fdata() > 0
    
    voxel_subject_matrix = []

    for sub_path in sub_maps:
        sub_img = nib.load(sub_path).get_fdata()
        roi_data = sub_img[roi_mask]
        
        roi_data = np.nan_to_num(roi_data, nan=0.0)  
        
        if roi_data.size == 0:
            voxel_subject_matrix.append(np.full(1, np.nan)) 
        else:
            voxel_subject_matrix.append(roi_data)

    # Transpose to shape (subjects, voxels)
    data = np.stack(voxel_subject_matrix, axis=1).T 

    # Compute ISC matrix
    isc_matrix = np.corrcoef(data)  # shape: (subjects, subjects)

    # Compute mean ISC per subject (excluding self-correlation)
    np.fill_diagonal(isc_matrix, np.nan)
    # apply fisher-z transformation
    isc_matrix = np.arctanh(isc_matrix)  # Fisher-z transformation
    
    mean_isc_per_subject = np.nanmean(isc_matrix, axis=1)

    # Store in dict
    isc_dict[roi_name] = mean_isc_per_subject

# === Combine all ISC values into a DataFrame ===
df_isc = pd.DataFrame(isc_dict)
df_isc.insert(0, "ID", subject_ids)

print(df_isc.head())

for net_label, roi_list in network_map.items():
       df_isc[net_label] = df_isc[roi_list].mean(axis=1)

# re-arrange order
df_isc = df_isc[desired_column_order] 

df_isc.to_csv(OUTPUT_FOLDER / f"{CON}_movieDM_ROI_acrosssubWithinMovie_n{n_subjects}.csv", index=False)


#### movie TP

In [ ]:
import os
import glob
import nibabel as nib
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr
from tqdm import tqdm

# === Set Paths ===
ROI_dir = 'unthres_ROI/'
ROIs = sorted(glob.glob(os.path.join(ROI_dir, 'rROI*.nii')))
ROI_names = ["_".join(os.path.splitext(roi)[0].split('_')[-2:]) for roi in ROIs]

ROOT_FOLDER = Path("movieTP_lexisurp_10feature_crossresult/")
CON = "GPT3_surprisal"
pattern = ROOT_FOLDER / "*" / f"*{CON}*.nii.gz"
sub_maps = sorted(glob.glob(str(pattern)))
n_subjects = len(sub_maps)
subject_ids = [os.path.basename(p).split("_")[0] for p in sub_maps]
print(f"Found {n_subjects} subject files.")

OUTPUT_FOLDER = Path("2_movieTP_ROIresults/cross_movie_similarity/")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# === Output container for ISC results ===
isc_dict = {}

# === Process Each ROI ===
for roi_path, roi_name in tqdm(zip(ROIs, ROI_names), total=len(ROIs), desc="Processing ROIs"):
    
    roi_mask = nib.load(roi_path).get_fdata() > 0
    
    voxel_subject_matrix = []

    for sub_path in sub_maps:
        sub_img = nib.load(sub_path).get_fdata()
        roi_data = sub_img[roi_mask]
        
        roi_data = np.nan_to_num(roi_data, nan=0.0)  
        
        if roi_data.size == 0:
            voxel_subject_matrix.append(np.full(1, np.nan))  
        else:
            voxel_subject_matrix.append(roi_data)

    # Transpose to shape (subjects, voxels)
    data = np.stack(voxel_subject_matrix, axis=1).T  

    # Compute ISC matrix
    isc_matrix = np.corrcoef(data)  

    # Compute mean ISC per subject (excluding self-correlation)
    np.fill_diagonal(isc_matrix, np.nan)
    # apply fisher-z transformation
    isc_matrix = np.arctanh(isc_matrix)  # Fisher-z transformation
    
    mean_isc_per_subject = np.nanmean(isc_matrix, axis=1)

    # Store in dict
    isc_dict[roi_name] = mean_isc_per_subject

# === Combine all ISC values into a DataFrame ===
df_isc = pd.DataFrame(isc_dict)
df_isc.insert(0, "ID", subject_ids)
print(df_isc.head())

for net_label, roi_list in network_map.items():
       df_isc[net_label] = df_isc[roi_list].mean(axis=1)

# re-arrange order
df_isc = df_isc[desired_column_order] 

df_isc.to_csv(os.path.join(OUTPUT_FOLDER, f"{CON}_movieTP_ROI_acrosssubWithinMovie_n{n_subjects}.csv"), index=False)